# Stock forecasting: inspect and benchmark the whole pipeline
This notebook uses the same public LSTMModel / GRUModel classes, transformations, metrics and artifact loader as command-line training. Your previous notebook is preserved as `Models_benchmark_legacy.ipynb`.
Run sequentially. Default: a short CPU experiment on local BIAT data, with no downloads or cloud operations. Change configuration before running. Historical benchmark results do not establish future profitability.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'stock_forecasting').is_dir())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace
from stock_forecasting.config import Config
from stock_forecasting.data import normalize, read_csv, features, targets, sequences, split_masks, fit_scaler, transform
from stock_forecasting.models import create_forecaster, ARIMAForecaster, naive_predictions
from stock_forecasting.evaluation import evaluate
from stock_forecasting.artifacts import save_bundle, load_bundle
from stock_forecasting.benchmark import split_metadata
from stock_forecasting.storage import LocalStore, snapshot
from src.prediction_model import LSTMModel, GRUModel
config = replace(Config.read(ROOT / 'configs/forecast.json'), epochs=3, device='cpu')
DATA = ROOT / 'data/raw' / f'{config.symbol}.csv'
config.to_dict()

{'symbol': 'BIAT',
 'horizon': 1,
 'sequence_length': 20,
 'feature_groups': ['price', 'volatility'],
 'family': 'lstm',
 'task': 'return',
 'hidden_size': 32,
 'layers': 1,
 'dropout': 0.1,
 'epochs': 3,
 'patience': 5,
 'batch_size': 64,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'loss': 'huber',
 'classification_weight': 0.5,
 'seed': 7,
 'device': 'cpu',
 'train_fraction': 0.6,
 'validation_fraction': 0.2,
 'adjustment': 'unverified',
 'calendar': None,
 'session_file': None,
 'as_of': '2026-10-09',
 'start': '2018-01-01',
 'end': '2026-10-07',
 'source': 'ilboursa',
 'cost_bps': 10.0,
 'threshold': 0.0}

## 1. Observations and data quality
Normalize column names and validate timestamps, OHLCV and price basis. Missing sessions are not interpolated. Without a verified exchange calendar, horizons count observed bars. Unverified corporate-action adjustments limit interpretation of returns.

In [2]:
raw = read_csv(DATA)
frame, quality = normalize(raw, config)
display(quality, frame.head(), frame.tail())
frame.plot(x='timestamp', y='close', figsize=(12, 3), title='Observed close')

{'symbol': 'BIAT',
 'source': 'ilboursa',
 'rows': 2187,
 'excluded_by_date': 1940,
 'start': '2018-01-02',
 'end': '2026-10-07',
 'zero_volume_rows': 0,
 'adjustment': 'unverified',
 'calendar': None,
 'horizon_basis': 'observed_bars_unverified_sessions',
 'timezone_convention': 'exchange-local session date; current UTC date excluded',
 'corporate_actions': 'provider adjustment required; no inferred split corrections'}

,symbol,timestamp,open,high,low,close,volume,raw_close,source
0,BIAT,2018-01-02,60.00,60.00,59.00,59.050,5794,59.050,ilboursa
1,BIAT,2018-01-03,59.00,60.00,58.70,60.000,1930,60.000,ilboursa
2,BIAT,2018-01-04,60.00,60.00,59.04,59.250,1440,59.250,ilboursa
3,BIAT,2018-01-05,59.25,60.00,59.25,59.950,3910,59.950,ilboursa
4,BIAT,2018-01-08,60.45,61.95,60.45,61.265,6162,61.265,ilboursa


,symbol,timestamp,open,high,low,close,volume,raw_close,source
2182,BIAT,2026-10-01,160.00,163.00,158.00,160.00,8085,160.00,ilboursa
2183,BIAT,2026-10-02,161.49,161.95,160.00,161.49,7890,161.49,ilboursa
2184,BIAT,2026-10-05,162.00,162.90,161.49,162.00,7125,162.00,ilboursa
2185,BIAT,2026-10-06,166.50,166.50,161.50,166.50,44302,166.50,ilboursa
2186,BIAT,2026-10-07,161.50,168.70,161.50,161.50,22050,161.50,ilboursa


<Axes: title={'center': 'Observed close'}, xlabel='timestamp'>

## 2. Causal features and future labels
Each input uses information through time t. The label is y = log(close[t+h] / close[t]); direction is y > 0. Future label columns are excluded from inputs. Rolling indicators require warmup; unavailable final labels are excluded from supervised samples.

In [3]:
engineered, columns = features(frame, config.feature_groups)
engineered = targets(engineered, config.horizon)
display(engineered[['timestamp', *columns, 'target_return', 'target_timestamp']].tail(8))
samples = sequences(engineered, columns, config.sequence_length, config.horizon)
print('X shape (samples, time steps, features):', samples.x.shape)
print('Feature order:', columns)

,timestamp,log_return,return_lag_1,return_lag_2,return_lag_5,momentum_5,ma_distance_5,momentum_20,ma_distance_20,intraday_range,open_close,overnight_gap,volatility_20,ewm_volatility,atr_14,high_low_volatility,parkinson_20,target_return,target_timestamp
2179,2026-09-28,-0.000252,0.000000,-0.013444,-0.015146,-0.022969,-0.006332,0.001954,-0.018528,0.002456,0.0,-0.000252,0.008818,0.009542,0.013129,0.009761,0.008001,-0.005113,2026-09-29
2180,2026-09-29,-0.005113,-0.000252,0.000000,-0.005554,-0.022529,-0.006926,-0.009323,-0.023087,0.011641,0.0,-0.005113,0.008770,0.009131,0.012948,0.009745,0.008110,0.000000,2026-09-30
2181,2026-09-30,0.000000,-0.005113,-0.000252,-0.003720,-0.018809,-0.003167,-0.016322,-0.022301,0.011337,0.0,0.000000,0.008595,0.008709,0.012920,0.009748,0.008174,0.012579,2026-10-01
2182,2026-10-01,0.012579,0.000000,-0.005113,-0.013444,0.007213,0.007988,-0.007472,-0.009558,0.031155,0.0,0.012579,0.009056,0.009366,0.014321,0.010673,0.009177,0.009269,2026-10-02
2183,2026-10-02,0.009269,0.012579,0.000000,0.000000,0.016483,0.014002,0.000557,-0.000362,0.012114,0.0,0.009269,0.009306,0.009387,0.011960,0.010430,0.009318,0.003153,2026-10-05
2184,2026-10-05,0.003153,0.009269,0.012579,-0.000252,0.019888,0.013146,0.003091,0.002640,0.008693,0.0,0.003153,0.009332,0.008967,0.011892,0.010199,0.009388,0.027399,2026-10-06
2185,2026-10-06,0.027399,0.003153,0.009269,-0.005113,0.052400,0.030335,0.030490,0.028899,0.030490,0.0,0.027399,0.011143,0.011733,0.011870,0.010987,0.010143,-0.030490,2026-10-07
2186,2026-10-07,-0.030490,0.027399,0.003153,0.000000,0.021910,-0.004917,0.012148,-0.002600,0.043617,0.0,-0.030490,0.012938,0.015082,0.014763,0.012943,0.011553,NaN,NaT


X shape (samples, time steps, features): (2147, 20, 16)
Feature order: ['log_return', 'return_lag_1', 'return_lag_2', 'return_lag_5', 'momentum_5', 'ma_distance_5', 'momentum_20', 'ma_distance_20', 'intraday_range', 'open_close', 'overnight_gap', 'volatility_20', 'ewm_volatility', 'atr_14', 'high_low_volatility', 'parkinson_20']


## 3. Chronological splits, purging and scaling
Training uses the first 60%, validation the next 20%, and test the final 20% by default. Labels crossing a split boundary are purged. Fit the scaler on unique training input rows only; apply that frozen transform to validation and test. Use validation to select models; inspect test once after choices are fixed.

In [4]:
train_end = int(len(engineered) * config.train_fraction)
val_end = int(len(engineered) * (config.train_fraction + config.validation_fraction))
train, val, test = split_masks(samples, train_end, val_end)
scaler, rows = fit_scaler(engineered, columns, samples, train, config.sequence_length)
x = transform(samples.x, scaler)
display(split_metadata(engineered, samples, (train, val, test), rows))
display(pd.DataFrame({'feature': columns, 'training_mean': scaler.mean_, 'training_scale': scaler.scale_}))

{'purge': 'label end strictly before next split origin',
 'scaler_last_input': '2023-03-28 00:00:00',
 'train_start': '2018-02-26 00:00:00',
 'train_end': '2023-03-28 00:00:00',
 'train_last_label': '2023-03-29 00:00:00',
 'train_n': 1272,
 'validation_start': '2023-03-30 00:00:00',
 'validation_end': '2024-12-25 00:00:00',
 'validation_last_label': '2024-12-30 00:00:00',
 'validation_n': 436,
 'test_start': '2024-12-31 00:00:00',
 'test_end': '2026-10-06 00:00:00',
 'test_last_label': '2026-10-07 00:00:00',
 'test_n': 437}

,feature,training_mean,training_scale
0,log_return,0.000300,0.012286
1,return_lag_1,0.000296,0.012285
2,return_lag_2,0.000273,0.012272
3,return_lag_5,0.000280,0.012277
4,momentum_5,0.001407,0.030024
5,ma_distance_5,0.000619,0.014475
6,momentum_20,0.005974,0.059942
7,ma_distance_20,0.002992,0.033340
8,intraday_range,0.013389,0.010673
9,open_close,-0.000756,0.010278


## 4. Build the recurrent models
An LSTM maintains a cell state and hidden state. Its input, forget and output gates control what is written, retained and exposed. A GRU uses reset and update gates with one hidden state. Both process each input window in time order; the last hidden state feeds dropout and linear output heads.
`from_config` explicitly opts the existing classes into return/direction tasks. The original constructor remains a legacy close-price adapter. Hidden size controls state width, layers control depth, dropout regularizes, and sequence length controls historical context.
Return training normalizes targets using training-only mean/std and uses Huber (robust to large residuals) or MSE. Direction uses binary cross entropy on logits; multitask combines both. AdamW, gradient clipping, validation-based learning-rate reduction and early stopping are shared. An untrained head is not exposed as a meaningful prediction.

In [5]:
models = {
    'LSTM': LSTMModel.from_config(len(columns), replace(config, family='lstm')),
    'GRU': GRUModel.from_config(len(columns), replace(config, family='gru')),
}
for name, model in models.items():
    print(name, model.network)
    print('Parameters:', model.metadata()['parameters'])

LSTM RecurrentNetwork(
  (encoder): LSTM(16, 32, batch_first=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (regression): Linear(in_features=32, out_features=1, bias=True)
  (direction): Linear(in_features=32, out_features=1, bias=True)
)
Parameters: 6466
GRU RecurrentNetwork(
  (encoder): GRU(16, 32, batch_first=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (regression): Linear(in_features=32, out_features=1, bias=True)
  (direction): Linear(in_features=32, out_features=1, bias=True)
)
Parameters: 4866


## 5. Train and inspect learning curves
The best validation-loss checkpoint is restored. These short runs demonstrate the pipeline; use multiple seeds and a larger budget for conclusions.

In [6]:
for name, model in models.items():
    model.fit(x[train], samples.y[train], x[val], samples.y[val])
    pd.DataFrame(model.history).plot(x='epoch', y=['train_loss', 'validation_loss'], title=name)
plt.show()

C:\Users\rezgu\AppData\Local\Temp\ipykernel_30252\3401747813.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Baselines and ARIMA
Zero return predicts no change. Historical mean and positive-class frequency use training labels only. ARIMA(p,d,q) combines lagged values, differencing and lagged innovations. For daily log returns, the shared ARIMA baseline selects its order by training AIC; parameters stay frozen while realized observations are appended through each forecast origin. Sum h daily return forecasts for the h-bar target.

In [7]:
predictions = {name: model.predict(x[test]) for name, model in models.items()}
predictions.update(naive_predictions(samples.y[train], int(test.sum())))
arima = ARIMAForecaster().fit(engineered.log_return.iloc[1:train_end].to_numpy())
predictions['ARIMA'] = {'return': arima.rolling_predict(engineered.log_return.to_numpy(), samples.origins[test], train_end-1, config.horizon), 'probability': None}
display(pd.DataFrame(arima.selection))

,order,aic,converged,warnings
0,"[0, 0, 0]",-7808.846754,True,[]
1,"[1, 0, 0]",-7814.973909,True,[]
2,"[0, 0, 1]",-7814.149759,True,[]
3,"[1, 0, 1]",-7815.095109,True,[]


## 7. Metrics and prediction plots
Let e = prediction - actual. MAE = mean(|e|); RMSE = sqrt(mean(e?)), emphasizing large errors. Both are in decimal log-return units; lower is better. R? compares squared errors with a constant predictor; negative is possible. Out-of-sample R? uses the stated training-mean or zero baseline. MASE scales MAE by training-label first differences. Bias is mean(e). Pearson measures linear association; Spearman measures rank association, not profit.
Direction accuracy is the correct-sign fraction. Precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = 2TP/(2TP+FP+FN). Balanced accuracy averages sensitivity and specificity. Compare with the majority-class baseline.
For probability models, Brier = mean((p - actual_direction)?) and log loss penalize wrong confidence; lower is better. ROC-AUC measures ranking (higher better, undefined for one-class truth); average precision summarizes precision-recall performance. Calibration compares predicted probability with observed frequency; ECE averages bin discrepancies. These probabilities are uncalibrated unless a separate calibration procedure is fitted.
MAPE = 100 ? mean(|price_error / actual_price|) is useful only with suitable nonzero price targets. Do not apply it to near-zero returns. Undefined metrics are returned as null, not invented values.

In [8]:
metrics = {name: evaluate(samples.y[test], pred, samples.y[train]) for name, pred in predictions.items()}
display(pd.json_normalize([{'model': name, **values} for name, values in metrics.items()]).set_index('model'))
plot = pd.DataFrame({'date': engineered.timestamp.iloc[samples.origins[test]].to_numpy(), 'actual': samples.y[test]})
for name, pred in predictions.items():
    if pred.get('return') is not None:
        plot[name] = pred['return']
plot.set_index('date')[['actual', 'LSTM', 'GRU', 'ARIMA']].plot(figsize=(12, 4))

,n,positive_fraction,training_positive_fraction,regression.mae,regression.mse,regression.rmse,regression.median_absolute_error,regression.r2,regression.oos_r2_train_mean,regression.oos_r2_zero,...,direction.tpr,direction.tnr,direction.confusion_matrix,direction.train_majority_accuracy,probability.brier,probability.log_loss,probability.roc_auc,probability.pr_auc_average_precision,probability.ece_10_equal_width,probability.calibration
model,,,,,,,,,,,,,,,,,,,,,
LSTM,437,0.423341,0.410377,0.006968,0.000132,0.011480,0.003909,-0.004253,0.003182,0.007426,...,0.616216,0.476190,"[[120, 132], [71, 114]]",0.576659,NaN,NaN,NaN,NaN,NaN,NaN
GRU,437,0.423341,0.410377,0.006994,0.000130,0.011415,0.003810,0.007120,0.014471,0.018666,...,0.724324,0.325397,"[[82, 170], [51, 134]]",0.576659,NaN,NaN,NaN,NaN,NaN,NaN
zero_return,437,0.423341,0.410377,0.006859,0.000133,0.011523,0.003876,-0.011766,-0.004275,0.000000,...,0.000000,1.000000,"[[252, 0], [185, 0]]",0.576659,NaN,NaN,NaN,NaN,NaN,NaN
historical_mean,437,0.423341,0.410377,0.006903,0.000132,0.011498,0.003854,-0.007459,0.000000,0.004257,...,1.000000,0.000000,"[[0, 252], [0, 185]]",0.576659,NaN,NaN,NaN,NaN,NaN,NaN
historical_probability,437,0.423341,0.410377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,1.000000,"[[252, 0], [185, 0]]",0.576659,0.244291,0.681694,0.5,0.423341,0.012964,"[{'bin': 4, 'count': 437, 'mean_probability': ..."
majority_direction,437,0.423341,0.410377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,1.000000,"[[252, 0], [185, 0]]",0.576659,NaN,NaN,NaN,NaN,NaN,NaN
ARIMA,437,0.423341,0.410377,0.006961,0.000131,0.011459,0.003767,-0.000626,0.006783,0.011011,...,0.805405,0.257937,"[[65, 187], [36, 149]]",0.576659,NaN,NaN,NaN,NaN,NaN,NaN


<Axes: xlabel='date'>

## 8. Trading simulation
Signals formed after close enter at the next open and hold h bars without overlapping trades. Costs are charged on entry and exit. Cumulative return compounds net bar returns; drawdown measures decline from the running equity peak. Turnover measures traded exposure. Annualization assumes the stated session frequency. These are strategy metrics, not forecast accuracy. Zero-volume execution periods invalidate this simple fill model; no result is fabricated.

In [9]:
from stock_forecasting.backtesting import backtest
for name in ('LSTM', 'GRU'):
    pred = predictions[name]
    signal = pred['return'] if pred['return'] is not None else pred['probability'] - 0.5
    try:
        result = backtest(engineered, samples.origins[test], signal, horizon=config.horizon, cost_bps=config.cost_bps)
        summary, curve, trades = result
        display(name, summary, pd.DataFrame(trades).head())
        curve.plot(x="timestamp", y="equity", title=f"{name}: net strategy equity")
    except ValueError as exc:
        print(name, 'simulation unavailable:', exc)

'LSTM'

{'gross_cumulative_return': 0.6028586272323411,
 'net_cumulative_return': -0.0176315249409289,
 'cagr': -0.010053882809371917,
 'annualized_volatility': 0.13429665698812399,
 'sharpe_zero_risk_free': -0.008504083029584704,
 'sortino_zero_target': -0.013305775243259464,
 'max_drawdown': -0.18151873829475806,
 'calmar': -0.05538757543062019,
 'win_rate': 0.3795918367346939,
 'average_trade_return': -8.206299399108625e-06,
 'profit_factor': 0.9976998000663246,
 'turnover': 490.0,
 'number_of_trades': 245,
 'transaction_cost_total_return_units': 0.49000000000000005,
 'exposure': 0.5619266055045872,
 'buy_hold_gross_return': 0.7211979111158449,
 'net_minus_buy_hold_gross': -0.7388294360567738,
 'holding_period_bars': 1,
 'signal_policy': 'nonoverlapping; enter next open, exit h opens later',
 'annual_periods_observed': 247.6656298600311}

,decision,entry,exit,side,net_trade_return
0,2025-01-07 00:00:00,2025-01-08 00:00:00,2025-01-09 00:00:00,1,0.000139
1,2025-01-08 00:00:00,2025-01-09 00:00:00,2025-01-10 00:00:00,1,0.001095
2,2025-01-09 00:00:00,2025-01-10 00:00:00,2025-01-13 00:00:00,1,-0.002000
3,2025-01-10 00:00:00,2025-01-13 00:00:00,2025-01-14 00:00:00,1,-0.001894
4,2025-01-13 00:00:00,2025-01-14 00:00:00,2025-01-15 00:00:00,1,0.008532


'GRU'

{'gross_cumulative_return': 0.8997284145979609,
 'net_cumulative_return': 0.03701456403616943,
 'cagr': 0.020860585287436617,
 'annualized_volatility': 0.15678512166584602,
 'sharpe_zero_risk_free': 0.20960581208548848,
 'sortino_zero_target': 0.33199379504310145,
 'max_drawdown': -0.23487628758945667,
 'calmar': 0.08881520353344101,
 'win_rate': 0.37623762376237624,
 'average_trade_return': 0.00019093532643495853,
 'profit_factor': 1.0517213966683778,
 'turnover': 606.0,
 'number_of_trades': 303,
 'transaction_cost_total_return_units': 0.6060000000000001,
 'exposure': 0.694954128440367,
 'buy_hold_gross_return': 0.7211979111158449,
 'net_minus_buy_hold_gross': -0.6841833470796754,
 'holding_period_bars': 1,
 'signal_policy': 'nonoverlapping; enter next open, exit h opens later',
 'annual_periods_observed': 247.6656298600311}

,decision,entry,exit,side,net_trade_return
0,2025-01-14 00:00:00,2025-01-15 00:00:00,2025-01-16 00:00:00,1,0.019160
1,2025-01-15 00:00:00,2025-01-16 00:00:00,2025-01-17 00:00:00,1,-0.003546
2,2025-01-16 00:00:00,2025-01-17 00:00:00,2025-01-20 00:00:00,1,-0.005614
3,2025-01-17 00:00:00,2025-01-20 00:00:00,2025-01-21 00:00:00,1,-0.002000
4,2025-01-21 00:00:00,2025-01-22 00:00:00,2025-01-23 00:00:00,1,-0.004062


## 9. Save and reload a complete research artifact
Weights, scaler, feature order, configuration and cutoffs travel together. This cell writes local research bundles; it does not promote them. Routine candidate training is `python -m scripts.train_model`. The full multi-seed research runner is `python -m stock_forecasting.cli benchmark`; both construct the same public classes.

In [10]:
manifest = snapshot(frame, quality, LocalStore(ROOT / 'artifacts/notebook/storage'))
bundle = save_bundle(models['LSTM'], scaler, columns, manifest,
                     split_metadata(engineered, samples, (train,val,test), rows), ROOT / 'artifacts/notebook/models')
restored, restored_scaler, metadata = load_bundle(bundle)
np.testing.assert_allclose(restored.predict(x[test])['return'], predictions['LSTM']['return']) if config.task != 'direction' else None
display(bundle, metadata['model']['config'])

WindowsPath('C:/Users/rezgu/OneDrive/Bureau/Stock-Price-Predictions-Model/artifacts/notebook/models/BIAT-lstm-return-h1-340008898ccd')

{'symbol': 'BIAT',
 'horizon': 1,
 'sequence_length': 20,
 'feature_groups': ['price', 'volatility'],
 'family': 'lstm',
 'task': 'return',
 'hidden_size': 32,
 'layers': 1,
 'dropout': 0.1,
 'epochs': 3,
 'patience': 5,
 'batch_size': 64,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'loss': 'huber',
 'classification_weight': 0.5,
 'seed': 7,
 'device': 'cpu',
 'train_fraction': 0.6,
 'validation_fraction': 0.2,
 'adjustment': 'unverified',
 'calendar': None,
 'session_file': None,
 'as_of': '2026-10-09',
 'start': '2018-01-01',
 'end': '2026-10-07',
 'source': 'ilboursa',
 'cost_bps': 10.0,
 'threshold': 0.0}